# NOTEBOOK 6 — UNSW-NB15
**Dataset:** 2.5M records | 49 features | 9 attack types + Normal
**Download:** https://research.unsw.edu.au/projects/unsw-nb15-dataset
**Files needed:**
- UNSW_NB15_training-set.csv (175,341 records)
- UNSW_NB15_testing-set.csv  (82,332 records)
**Upload to DBFS:** /FileStore/malware/datasets/unswnb15/

**Attack categories:** Normal, Generic, Exploits, Fuzzers, DoS, Reconnaissance, Analysis, Backdoors, Shellcode, Worms

In [ ]:
# Cell 1 — Install libraries
%pip install xgboost lightgbm shap

In [ ]:
# Cell 2 — Imports
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

print('All imports OK')

In [ ]:
# Cell 3 — Load pre-split train/test CSVs
UNSW_PATH = '/dbfs/FileStore/malware/datasets/unswnb15/'

train_df = pd.read_csv(UNSW_PATH + 'UNSW_NB15_training-set.csv')
test_df  = pd.read_csv(UNSW_PATH + 'UNSW_NB15_testing-set.csv')

print(f'Train: {train_df.shape}')
print(f'Test:  {test_df.shape}')
print(f'\nColumns: {list(train_df.columns)}')
print(f'\nAttack type distribution (train):')
print(train_df['attack_cat'].value_counts())

In [ ]:
# Cell 4 — Clean and encode features

LABEL_BINARY = 'label'       # 0=normal, 1=attack
LABEL_MULTI  = 'attack_cat'  # 10 categories

# Columns to drop (IPs, ports, timestamps — not useful as raw features)
DROP_COLS = ['id', 'srcip', 'dstip', 'sport', 'dsport', 'Stime', 'Ltime']

def prepare_features(df):
    df = df.copy()

    # Encode categorical columns
    for col in ['proto', 'state', 'service']:
        if col in df.columns:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))

    # Drop ID/IP/timestamp columns
    df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors='ignore')

    # Replace inf with NaN then fill with 0
    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

    return df

train_clean = prepare_features(train_df)
test_clean  = prepare_features(test_df)

feature_cols = [c for c in train_clean.columns
                if c not in [LABEL_BINARY, LABEL_MULTI]]

print(f'Feature columns ({len(feature_cols)}): {feature_cols}')

In [ ]:
# Cell 5 — Build feature arrays for binary and multi-class

X_tr = train_clean[feature_cols].values.astype(np.float32)
X_te = test_clean[feature_cols].values.astype(np.float32)

# Binary labels
y_tr_bin = train_clean[LABEL_BINARY].values
y_te_bin = test_clean[LABEL_BINARY].values

# Multi-class labels
le_cat   = LabelEncoder()
y_tr_mc  = le_cat.fit_transform(train_clean[LABEL_MULTI].astype(str))
y_te_mc  = le_cat.transform(test_clean[LABEL_MULTI].astype(str))
attack_names = list(le_cat.classes_)

print(f'Train: {X_tr.shape} | Attack ratio: {y_tr_bin.mean():.2%}')
print(f'Test:  {X_te.shape} | Attack ratio: {y_te_bin.mean():.2%}')
print(f'Attack categories ({len(attack_names)}): {attack_names}')

In [ ]:
# Cell 6 — Train Binary XGBoost
mlflow.set_experiment('/malware-detection')

with mlflow.start_run(run_name='UNSWNB15_Binary_XGBoost'):
    params = {
        'n_estimators':  300,
        'max_depth':     6,
        'learning_rate': 0.1,
        'n_jobs':        -1,
        'random_state':  42
    }
    mlflow.log_params(params)
    mlflow.log_param('dataset',    'UNSW-NB15')
    mlflow.log_param('task',       'binary_normal_vs_attack')
    mlflow.log_param('model_type', 'XGBoost')

    model_bin = xgb.XGBClassifier(
        **params,
        use_label_encoder=False,
        eval_metric='logloss'
    )
    model_bin.fit(X_tr, y_tr_bin)

    y_prob = model_bin.predict_proba(X_te)[:, 1]
    auc    = roc_auc_score(y_te_bin, y_prob)

    mlflow.log_metric('roc_auc', auc)
    mlflow.sklearn.log_model(model_bin, 'model',
        registered_model_name='unsw_xgb_binary')
    print(f'UNSW-NB15 Binary AUC: {auc:.4f}  <- Expected ~0.98')

In [ ]:
# Cell 7 — Train Multi-class LightGBM (9 attack types)
with mlflow.start_run(run_name='UNSWNB15_MultiClass_LightGBM'):
    mlflow.log_param('dataset',      'UNSW-NB15')
    mlflow.log_param('task',         'multiclass_9_attack_types')
    mlflow.log_param('model_type',   'LightGBM')
    mlflow.log_param('class_names',  str(attack_names))

    model_mc = lgb.LGBMClassifier(
        objective='multiclass',
        num_class=len(attack_names),
        n_estimators=300,
        num_leaves=63,
        learning_rate=0.1,
        n_jobs=-1,
        random_state=42
    )
    model_mc.fit(X_tr, y_tr_mc)

    y_pred = model_mc.predict(X_te)
    report = classification_report(y_te_mc, y_pred,
                 target_names=attack_names, output_dict=True)

    mlflow.log_metric('accuracy', report['accuracy'])
    mlflow.log_metric('macro_f1', report['macro avg']['f1-score'])
    mlflow.sklearn.log_model(model_mc, 'model',
        registered_model_name='unsw_lgb_multiclass')

    print(classification_report(y_te_mc, y_pred, target_names=attack_names))

In [ ]:
# Cell 8 — Isolation Forest (Zero-day / anomaly detection)
# Trains ONLY on normal traffic — no attack labels needed
# This simulates detecting unknown/novel attacks

with mlflow.start_run(run_name='UNSWNB15_IsolationForest_Anomaly'):
    mlflow.log_param('dataset',    'UNSW-NB15')
    mlflow.log_param('task',       'anomaly_detection_zero_day_simulation')
    mlflow.log_param('model_type', 'IsolationForest')
    mlflow.log_param('note',       'Trained on normal traffic only')

    # Use ONLY normal (benign) samples for training
    X_normal = X_tr[y_tr_bin == 0]
    print(f'Training on {len(X_normal):,} normal samples only')

    iso = IsolationForest(
        n_estimators=200,
        contamination=0.05,
        n_jobs=-1,
        random_state=42
    )
    iso.fit(X_normal)

    # Predict: -1=anomaly, 1=normal → convert to 0/1
    preds     = iso.predict(X_te)
    preds_bin = (preds == -1).astype(int)

    auc = roc_auc_score(y_te_bin, preds_bin)
    mlflow.log_metric('roc_auc_anomaly', auc)
    mlflow.sklearn.log_model(iso, 'model',
        registered_model_name='unsw_isolation_forest')

    print(f'Isolation Forest AUC: {auc:.4f}')
    print('Note: Lower AUC expected (~0.75) — this is unsupervised, no labels used in training')

In [ ]:
# Cell 9 — Confusion matrix for binary model
y_pred_bin = model_bin.predict(X_te)
cm         = confusion_matrix(y_te_bin, y_pred_bin)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
    xticklabels=['Normal', 'Attack'],
    yticklabels=['Normal', 'Attack'])
ax.set_title('UNSW-NB15 Confusion Matrix — Binary', fontsize=14, fontweight='bold')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.savefig('/tmp/unsw_confusion.png', dpi=120)
display(fig)

In [ ]:
# Cell 10 — SHAP analysis
explainer = shap.TreeExplainer(model_bin)
shap_vals = explainer.shap_values(X_te[:500])

shap.summary_plot(shap_vals, X_te[:500],
    feature_names=feature_cols, max_display=20, show=False)
plt.savefig('/tmp/unsw_shap.png', bbox_inches='tight', dpi=120)
plt.close()
display(plt.imread('/tmp/unsw_shap.png'))

In [ ]:
# Cell 11 — Export all models for Jupyter app
import json
os.makedirs('/dbfs/FileStore/models', exist_ok=True)

joblib.dump(model_bin, '/dbfs/FileStore/models/unsw_xgb_binary.pkl')
joblib.dump(model_mc,  '/dbfs/FileStore/models/unsw_lgb_multiclass.pkl')
joblib.dump(iso,       '/dbfs/FileStore/models/unsw_isolation_forest.pkl')

meta = {
    'feature_cols':  feature_cols,
    'attack_names':  attack_names,
    'binary_labels': {0: 'Normal', 1: 'Attack'}
}
json.dump(meta, open('/dbfs/FileStore/models/unsw_metadata.json', 'w'), indent=2)

print('Exported:')
print('  /FileStore/models/unsw_xgb_binary.pkl')
print('  /FileStore/models/unsw_lgb_multiclass.pkl')
print('  /FileStore/models/unsw_isolation_forest.pkl')
print('  /FileStore/models/unsw_metadata.json')